# Phase 0: YOLO Training Notebook

This notebook trains YOLOv8n for Indian food detection.
- **Task**: detection only (no segmentation)
- **Model**: yolov8n
- **Data**: indianfoodnet_yolo
- **GPU settings**: batch=8, amp=True, workers=2, device=0

In [ ]:
# Cell 1: GPU check + imports
import torch
import yaml
import os
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt

print("=== GPU Check ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA not available - training will be very slow!")

# Change to project root
project_root = Path.cwd()
while not (project_root / "data").exists():
    project_root = project_root.parent
os.chdir(project_root)
print(f"Working directory: {project_root}")

In [ ]:
# Cell 2: YOLO detection training
print("=== Starting YOLOv8n Training ===")

# Load YOLOv8n model for detection
model = YOLO('yolov8n.pt')  # detection model

# Training configuration for 4GB VRAM
training_args = {
    'data': 'data/indianfoodnet_yolo/data.yaml',
    'epochs': 60,
    'imgsz': 640,
    'batch': 8,  # Critical for 4GB VRAM
    'patience': 15,
    'device': 0,  # GPU
    'workers': 2,  # Reduce memory usage
    'amp': True,  # Mixed precision
    'save_period': 10,  # Save every 10 epochs
    'project': 'models/runs',
    'name': 'indian_food_detection'
}

print(f"Training args: {training_args}")

# Train the model
results = model.train(**training_args)

print("✅ Training completed!")

In [ ]:
# Cell 3: Copy best.pt → models/yolov8n_indian.pt
import shutil

print("=== Saving Model ===")

# Find the best model from training
runs_dir = Path('models/runs/indian_food_detection')
if runs_dir.exists():
    # Find the most recent run directory
    run_dirs = [d for d in runs_dir.iterdir() if d.is_dir()]
    if run_dirs:
        latest_run = max(run_dirs, key=lambda x: x.stat().st_mtime)
        best_model_path = latest_run / 'weights' / 'best.pt'
        
        if best_model_path.exists():
            # Create models directory if it doesn't exist
            Path('models').mkdir(exist_ok=True)
            
            # Copy best model to models/yolov8n_indian.pt
            shutil.copy2(best_model_path, 'models/yolov8n_indian.pt')
            print(f"✅ Copied best model to: models/yolov8n_indian.pt")
            print(f"Original path: {best_model_path}")
        else:
            print(f"❌ best.pt not found in {latest_run / 'weights'}")
    else:
        print("❌ No run directories found")
else:
    print("❌ Training runs directory not found")

In [ ]:
# Cell 4: Save class names → models/class_names.json
import json

print("=== Extracting Class Names ===")

# Load data.yaml to get class names
data_yaml_path = 'data/indianfoodnet_yolo/data.yaml'
if Path(data_yaml_path).exists():
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    class_names = data_config.get('names', [])
    print(f"Found {len(class_names)} classes: {class_names}")
    
    # Save class names to JSON
    with open('models/class_names.json', 'w') as f:
        json.dump(class_names, f, indent=2)
    
    print(f"✅ Saved class names to: models/class_names.json")
else:
    print(f"❌ data.yaml not found at {data_yaml_path}")

In [ ]:
# Cell 5: Plot training loss curves inline
print("=== Plotting Training Curves ===")

# Find results.csv from training
runs_dir = Path('models/runs/indian_food_detection')
if runs_dir.exists():
    run_dirs = [d for d in runs_dir.iterdir() if d.is_dir()]
    if run_dirs:
        latest_run = max(run_dirs, key=lambda x: x.stat().st_mtime)
        results_csv = latest_run / 'results.csv'
        
        if results_csv.exists():
            # Read training results
            import pandas as pd
            df = pd.read_csv(results_csv)
            
            # Plot training curves
            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            fig.suptitle('YOLO Training Progress', fontsize=16)
            
            # Loss curves
            axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss')
            axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss')
            axes[0, 0].set_title('Box Loss')
            axes[0, 0].legend()
            
            axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train Cls Loss')
            axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val Cls Loss')
            axes[0, 1].set_title('Classification Loss')
            axes[0, 1].legend()
            
            # Metrics
            axes[1, 0].plot(df['epoch'], df['metrics/precision'], label='Precision')
            axes[1, 0].plot(df['epoch'], df['metrics/recall'], label='Recall')
            axes[1, 0].set_title('Precision & Recall')
            axes[1, 0].legend()
            
            axes[1, 1].plot(df['epoch'], df['metrics/mAP50'], label='mAP@0.5')
            axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95'], label='mAP@0.5:0.95')
            axes[1, 1].set_title('Mean Average Precision')
            axes[1, 1].legend()
            
            plt.tight_layout()
            plt.show()
            
            # Print final metrics
            final_metrics = df.iloc[-1]
            print(f"\n=== Final Training Metrics ===")
            print(f"Final mAP@0.5: {final_metrics['metrics/mAP50']:.4f}")
            print(f"Final mAP@0.5:0.95: {final_metrics['metrics/mAP50-95']:.4f}")
            print(f"Final Precision: {final_metrics['metrics/precision']:.4f}")
            print(f"Final Recall: {final_metrics['metrics/recall']:.4f}")
        else:
            print(f"❌ results.csv not found in {latest_run}")
    else:
        print("❌ No run directories found")
else:
    print("❌ Training runs directory not found")